# 🌊 Interactive Flood Prediction Explorer (PyGWalker)
Interactive test set prediction explorer. Filter test samples by `risk_level`, sort by `flood_probability`, and evaluate actual ground truth.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import pygwalker as pyg

# Load processed test data and v3 model
X_test = np.load('../data/processed/X_test_v3.npy')
y_test = np.load('../data/processed/y_test.npy')
model_data = joblib.load('../models/xgboost_flood_model_v3.pkl')
model = model_data['model']
feature_names = model_data['feature_names']
threshold = model_data.get('optimal_threshold', 0.35)

# Build prediction DataFrame
df_test = pd.DataFrame(X_test, columns=feature_names)
probas = model.predict_proba(X_test)[:, 1]
preds = (probas >= threshold).astype(int)

df_test['actual_flooded'] = y_test
df_test['predicted_flooded'] = preds
df_test['flood_probability'] = np.round(probas, 4)
df_test['risk_level'] = pd.cut(
    probas,
    bins=[-0.01, 0.25, 0.50, 0.75, 1.01],
    labels=['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
)

print(f'Test Dataset Predictions ({len(df_test)} samples):')
display(df_test.head(10))

# Launch Interactive PyGWalker Exploration GUI
walker = pyg.walk(df_test)
